# Issue 22 — independent Development comparisons

Technical smoke only: one prefix-selected Development entity per dataset/city,
main seed42, budgets7/14/28, 43 forecast days. Source ensembles use42/43/44;
this is not three-seed replication of the comparison. No Final data or tuning.

Run on Colab GPU. UrbanEV and each CHARGED city have separate Source training.
Example thresholds are unvalidated; read docs/data/BASELINE_CONTRACT.md.
Respect runtime quotas. Persist artifacts instead of rotating accounts.


In [ ]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import zipfile

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch

assert torch.cuda.is_available(), "Select a GPU runtime; do not run long training locally"
assert torch.__version__.split("+")[0] == "2.11.0", "Review runtime compatibility explicitly"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "numpy==2.3.3", "ruff==0.17.0", "gdown"], check=True
)
print(sys.version, torch.__version__, torch.cuda.get_device_name(0))


def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def extract(archive, destination):
    with zipfile.ZipFile(archive) as z:
        assert all(
            not p.startswith("/") and ".." not in pathlib.PurePosixPath(p).parts
            for p in z.namelist()
        )
        z.extractall(destination)

In [ ]:
RUNTIME_SHA = "cceab803f6898d1b1192edc74d49a762670e426b"
REPO = pathlib.Path("/content/ev-issue22-reproduction")
assert not REPO.exists()
subprocess.run(
    ["git", "clone", "https://github.com/HungLD9505/adaptive-retention-ev.git", str(REPO)],
    check=True,
)
subprocess.run(["git", "checkout", "--detach", RUNTIME_SHA], cwd=REPO, check=True)
assert (
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
    == RUNTIME_SHA
)
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=REPO, check=True
)
subprocess.run([sys.executable, "-m", "ruff", "check", "."], cwd=REPO, check=True)

In [ ]:
import stat

import gdown

catalog = json.loads((REPO / "data/releases/DATA-v1-20261019.json").read_text())
packages = [e for e in catalog.get("packages", catalog.get("files", [])) if e["kind"] == "inputs"]
DATA_ROOT = pathlib.Path("/content/issue22-reproduction-data")
assert not DATA_ROOT.exists(), "Use a fresh extraction destination"
for entry in packages:
    archive = pathlib.Path("/content") / entry["name"]
    if not archive.exists():
        gdown.download(id=entry["drive_file_id"], output=str(archive), quiet=False)
    assert sha(archive) == entry["sha256"]
    with zipfile.ZipFile(archive) as z:
        assert all(
            not p.filename.startswith("/")
            and ".." not in pathlib.PurePosixPath(p.filename).parts
            and stat.S_IFMT(p.external_attr >> 16) != stat.S_IFLNK
            for p in z.infolist()
        )
        assert all(not (DATA_ROOT / p.filename).exists() for p in z.infolist() if not p.is_dir())
        z.extractall(DATA_ROOT)
    print(entry["name"], entry["sha256"])
INPUTS = DATA_ROOT / "data/inputs/issue18-input-handoff-v1-20261019"

## Sequential matrix

A clean reproduction trains three Source models per namespace for one epoch.
The original UrbanEV execution reused the exact #19 seed42 checkpoint and v1
seed43/44 checkpoints, with hashes recorded in its matrix. Do not claim a clean
retraining has identical bytes across a different runtime. Namespace outputs must
be new. Every completed cell contains its own daily transactions and run report.


In [ ]:
OUTPUT = pathlib.Path("/content/issue22-development-reproduction")
OUTPUT.mkdir(exist_ok=False)
for namespace in (
    "urbanev-SZH",
    "charged-AMS",
    "charged-JHB",
    "charged-LOA",
    "charged-MEL",
    "charged-SPO",
    "charged-SZH",
):
    destination = OUTPUT / namespace
    command = [
        sys.executable,
        "-m",
        "tools.comparison_matrix",
        "--inputs-root",
        str(INPUTS),
        "--output",
        str(destination),
        "--namespaces",
        namespace,
        "--seeds",
        "42",
        "--days",
        "43",
    ]
    with (OUTPUT / (namespace + ".log")).open("x") as stream:
        subprocess.run(command, cwd=REPO, stdout=stream, stderr=subprocess.STDOUT, check=True)
    result = json.loads((destination / "matrix.json").read_text())
    assert result["status"] == "technical_success"
    print(namespace, len(result["runs"]), result["matrix_wall_seconds"])

## Preserve results

Copy complete output directories, logs, exact code and pip-freeze output into new
Drive artifacts. Record sizes and SHA-256; verify copied bytes before disconnecting.
Never overwrite earlier runs. Resume an interrupted cell through the runner's
matching-binding daily checkpoint interface; do not present a partial matrix as
complete. The orchestration CLI itself does not skip/recover a partially existing
output folder automatically. Independent configuration review is required before
any Final execution.
